# Project 2 Part I (Core)

*Project 2: Initial Analysis and Problem Selection*

---

## Objective

Perform an initial Exploratory Data Analysis (EDA) on at least four datasets, identify and diagnose potential issues, and select a specific problem to address (regression, classification, clustering, or forecasting). Submit a repository containing the selected dataset, the initial EDA, and a description of the chosen problem.

---

# Part I: Dataset Search and Analysis

## Problem Statement

Develop a predictive model to estimate individual calorie needs and identify nutritional risk patterns to support preventive healthcare and personalized dietary recommendations.

# Dataset Description

This dataset contains demographic, biometric, lifestyle, and dietary information, including age, gender, height, weight, physical activity, dietary habits, and calorie intake.

## Dataset Source

This project uses the Healthy Diet and Calorie Intake dataset from Kaggle. The dataset contains records for 6,000 individuals and includes 15 features related to demographics, lifestyle, diet, and health. Variables include age, gender, height, weight, BMI, activity level, diet type, protein, carbohydrate and fat intake, water consumption, daily calorie needs and intake, and health status.

The dataset is available at: [https://www.kaggle.com/datasets/aliyasaly1231/healthy-diet-and-calorie-intake](https://www.kaggle.com/datasets/aliyasaly1231/healthy-diet-and-calorie-intake).

## Dataset Information

- Number of observations: **6000**
- Number of features: **15**
- Target variable: `Health_Status`

## Data Dictionary

| Column Name                   | Data Type                | Description                                                                                                                                          | Example     |
| ----------------------------- | ------------------------ | ---------------------------------------------------------------------------------------------------------------------------------------------------- | ----------- |
| **Person_ID**                 | String                   | Unique identifier for each individual.                                                                              | P0001       |
| **Age**                       | Integer                  | Age of the individual in years.                                                                                                                      | 50          |
| **Gender**                    | String                   | Gender of the individual. Possible values: Male, Female, Other.                                                                                      | Male        |
| **Height_cm**                 | Float                    | Height measured in centimeters. Used in BMI calculation.                                                                                             | 176.4       |
| **Weight_kg**                 | Float                    | Body weight measured in kilograms.                                                                                                                   | 74.8        |
| **BMI**                       | Float                    | Body Mass Index calculated as weight (kg) / height (m)². Used to assess weight status.                                                               | 24.0        |
| **Activity_Level**            | String                   | Physical activity category. Values include Sedentary, Lightly Active, Moderately Active, Very Active, and Athlete.                                   | Very Active |
| **Daily_Calorie_Requirement** | Integer                  | Estimated daily calorie requirement based on age, gender, height, weight, and activity level (TDEE).                                                 | 2852        |
| **Daily_Calorie_Consumed**    | Integer                  | Actual number of calories consumed per day.                                                                                                          | 2625        |
| **Protein_Intake_g**          | Float                    | Daily protein intake measured in grams.                                                                                                              | 183.0       |
| **Carbohydrate_Intake_g**     | Float                    | Daily carbohydrate intake measured in grams.                                                                                                         | 16.9        |
| **Fat_Intake_g**              | Float                    | Daily fat intake measured in grams.                                                                                                                  | 202.8       |
| **Water_Intake_Liters**       | Float                    | Daily water consumption measured in liters.                                                                                                          | 3.3         |
| **Diet_Type**                 | String                   | Dietary pattern followed by the individual. Values include Keto, Vegan, Balanced, Mediterranean, High Protein, and Vegetarian.                       | Keto        |
| **Health_Status**             | String (Target Variable) | Health classification of the individual. Categories: Healthy, Overweight, Obese, Underweight.  | Healthy     |

**Note**

**TDEE** is **Total Daily Energy Expenditure**.

It is the estimated number of calories a person burns in a day, including:

- **Basal Metabolic Rate (BMR):** Calories needed for basic body functions (breathing, circulation, etc.).
- **Physical Activity:** Calories burned through exercise and daily movement.
- **Thermic Effect of Food (TEF):** Calories used to digest and process food.

In this dataset, **Daily_Calorie_Requirement** represents the individual's TDEE, which is calculated based on factors such as age, gender, height, weight, and activity level.

Example:

- TDEE = 2,500 calories/day
- Calories consumed = 2,200 calories/day
- Result = **300-calorie deficit** (may lead to weight loss if sustained)

Or:

- TDEE = 2,500 calories/day
- Calories consumed = 2,800 calories/day
- Result = **300-calorie surplus** (may lead to weight gain if sustained)

---

# Data Loading and Inspection

## Import Libraries

In [ ]:
# Standard library imports
import math
from pathlib import Path

# Third-party imports
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from scipy.stats import zscore

### Dataset Availability and Path Setup

In [4]:
data_dir = Path("../datasets")

# List available files (for verification)
available_files = list(data_dir.iterdir())
print("Available files:", available_files)
file_name = "3_healthy_diet.csv"
dataset_path = data_dir / file_name

Available files: [WindowsPath('../datasets/1_aircrafts_chile_30042026.csv'), WindowsPath('../datasets/2_earthquakes_chile.csv'), WindowsPath('../datasets/3_healthy_diet.csv'), WindowsPath('../datasets/4_temp_chile_2014.csv'), WindowsPath('../datasets/README.md')]


In [5]:
# Validate dataset existence
if not dataset_path.exists():
    raise FileNotFoundError(f"{file_name} not found in {data_dir}")

print("Dataset found at:", dataset_path)

Dataset found at: ..\datasets\3_healthy_diet.csv


## Load Dataset

In [6]:
df = pd.read_csv(dataset_path)

## Dataset Overview


### First and Last Records

In [7]:
df.head()

,Person_ID,Age,Gender,Height_cm,Weight_kg,BMI,Activity_Level,Daily_Calorie_Requirement,Daily_Calorie_Consumed,Protein_Intake_g,Carbohydrate_Intake_g,Fat_Intake_g,Water_Intake_Liters,Diet_Type,Health_Status
0,P0001,50,Male,176.4,74.8,24.0,Very Active,2852,2625,183.0,16.9,202.8,3.3,Keto,Healthy
1,P0002,18,Female,167.6,75.5,26.9,Sedentary,1904,2044,90.1,306.5,50.8,1.9,Vegan,Overweight
2,P0003,68,Female,161.9,87.2,33.3,Lightly Active,2009,2540,222.7,281.3,58.2,2.4,High Protein,Obese
3,P0004,22,Female,169.3,66.9,23.3,Moderately Active,2318,2096,69.5,299.8,68.7,2.9,Balanced,Healthy
4,P0005,30,Male,179.1,75.3,23.5,Sedentary,2144,1937,32.9,285.6,73.7,2.2,Balanced,Healthy


In [8]:
df.tail()

,Person_ID,Age,Gender,Height_cm,Weight_kg,BMI,Activity_Level,Daily_Calorie_Requirement,Daily_Calorie_Consumed,Protein_Intake_g,Carbohydrate_Intake_g,Fat_Intake_g,Water_Intake_Liters,Diet_Type,Health_Status
5995,P5996,45,Female,161.1,79.1,30.5,Lightly Active,2039,2296,85.0,305.9,81.4,2.4,Balanced,Obese
5996,P5997,53,Female,157.7,63.7,25.6,Sedentary,1555,1851,177.0,179.7,47.1,1.8,High Protein,Overweight
5997,P5998,43,Female,154.0,81.8,34.5,Athlete,2840,3601,139.7,483.3,123.2,4.4,Mediterranean,Obese
5998,P5999,51,Female,162.8,77.8,29.4,Lightly Active,1994,2126,118.7,51.9,160.4,2.8,Keto,Overweight
5999,P6000,25,Male,171.5,75.9,25.8,Athlete,3394,4001,163.3,623.0,95.1,4.4,Vegan,Overweight


### Verify Structure

In [9]:
print(f"{file_name}")
print(f"Shape: {df.shape}")
print(f"→ {df.shape[0]} rows, {df.shape[1]} columns\n")

3_healthy_diet.csv
Shape: (6000, 15)
→ 6000 rows, 15 columns



---

## Initial Data Inspection

### Inspect Data Types

In [10]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 6000 entries, 0 to 5999
Data columns (total 15 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   Person_ID                  6000 non-null   str    
 1   Age                        6000 non-null   int64  
 2   Gender                     6000 non-null   str    
 3   Height_cm                  6000 non-null   float64
 4   Weight_kg                  6000 non-null   float64
 5   BMI                        6000 non-null   float64
 6   Activity_Level             6000 non-null   str    
 7   Daily_Calorie_Requirement  6000 non-null   int64  
 8   Daily_Calorie_Consumed     6000 non-null   int64  
 9   Protein_Intake_g           6000 non-null   float64
 10  Carbohydrate_Intake_g      6000 non-null   float64
 11  Fat_Intake_g               6000 non-null   float64
 12  Water_Intake_Liters        6000 non-null   float64
 13  Diet_Type                  6000 non-null   str    
 14  Hea

In [11]:
df.columns.tolist()

['Person_ID',
 'Age',
 'Gender',
 'Height_cm',
 'Weight_kg',
 'BMI',
 'Activity_Level',
 'Daily_Calorie_Requirement',
 'Daily_Calorie_Consumed',
 'Protein_Intake_g',
 'Carbohydrate_Intake_g',
 'Fat_Intake_g',
 'Water_Intake_Liters',
 'Diet_Type',
 'Health_Status']

### Descriptive Statistics

In [12]:
df.describe().T

,count,mean,std,min,25%,50%,75%,max
Age,6000.0,48.817167,18.141837,18.0,33.0,48.00,65.000,80.0
Height_cm,6000.0,168.592533,9.160213,145.0,161.7,168.20,175.200,198.6
Weight_kg,6000.0,74.070933,12.417877,40.0,65.5,74.10,82.700,116.6
BMI,6000.0,26.094300,4.162643,12.9,23.3,26.00,28.900,44.0
Daily_Calorie_Requirement,6000.0,2286.223833,469.541134,1223.0,1941.0,2232.50,2581.250,4079.0
Daily_Calorie_Consumed,6000.0,2490.520333,602.279679,1000.0,2057.0,2433.00,2859.000,5158.0
Protein_Intake_g,6000.0,123.773150,61.999158,17.4,78.8,108.90,155.100,457.5
Carbohydrate_Intake_g,6000.0,288.035317,123.914738,-23.6,229.1,297.15,366.125,729.7
Fat_Intake_g,6000.0,93.698417,43.953993,23.2,65.9,82.30,105.100,341.7
Water_Intake_Liters,6000.0,2.734917,0.681486,1.5,2.2,2.60,3.100,5.0


The dataset contains information for individuals aged **18 to 80 years**, with an average age of approximately **49 years**, representing a broad range of age groups.

Participants' heights range from **145.0 cm to 198.6 cm**, with an average height of approximately **168.6 cm**.

Weights range from **40.0 kg to 116.6 kg**, with an average of approximately **74.1 kg**, indicating substantial variation in body composition across individuals.

BMI values range from **12.9 to 44.0**, with an average BMI of approximately **26.1**. According to WHO classifications, this average falls within the **overweight** category, suggesting that a significant portion of the population may be above the healthy weight range.

The average **Daily Calorie Requirement** is approximately **2,286 kcal**, while the average **Daily Calorie Consumed** is approximately **2,491 kcal**. This indicates an average calorie surplus of about **204 kcal per day**, which may help explain the relatively high average BMI observed in the dataset.

Protein, carbohydrate, and fat intake show considerable variability, reflecting the presence of multiple dietary patterns. However, the minimum value of **-23.6 g** for `Carbohydrate_Intake_g` is physiologically impossible and should be investigated as a potential data quality issue.

Daily water intake ranges from **1.5 to 5.0 liters**, with an average of approximately **2.7 liters**, which is within commonly recommended hydration ranges for adults.

Overall, the wide variation across demographic, dietary, and health-related variables makes this dataset suitable for exploring relationships between nutrition, lifestyle factors, calorie balance, and health status. However, potential anomalies such as negative carbohydrate intake should be addressed during data cleaning and preprocessing.


In [13]:
df.describe(include="str").T

,count,unique,top,freq
Person_ID,6000,6000,P0001,1
Gender,6000,3,Male,2892
Activity_Level,6000,5,Lightly Active,1627
Diet_Type,6000,6,Balanced,1536
Health_Status,6000,4,Overweight,2500


- **Person_ID** contains 6,000 unique values, confirming that each record represents a different individual. As an identifier, it will not be used for predictive analysis.

- **Gender** has 3 categories, with **Male** being the most frequent (2,892 records). The distribution of the remaining categories will be analyzed later.

- **Activity_Level** contains 5 categories, with **Lightly Active** being the most common (1,627 records).

- **Diet_Type** contains 6 dietary patterns, with **Balanced** being the most frequent diet (1,536 records).

- **Health_Status** contains 4 categories, with **Overweight** being the most common class (2,500 records), suggesting a potential class imbalance that should be considered during model development.

- No categorical variable contains missing values, indicating complete records across all categories.

### Comparison with Real Standard

In [14]:
# Mean BMI by Health Status
bmi_mean = (
    df.groupby('Health_Status')['BMI']
    .mean()
    .round(2)
    .reset_index()
    .rename(columns={'BMI': 'Mean_BMI'})
)

bmi_mean

,Health_Status,Mean_BMI
0,Healthy,22.52
1,Obese,32.26
2,Overweight,27.29
3,Underweight,17.03


| Health_Status | Dataset Mean BMI | WHO BMI Range |
| ------------- | ---------------: | ----------------: |
| Underweight   |            17.03 |              < 18.5 |
| Healthy       |            22.52 |              18.5 – 24.9 |
| Overweight    |            27.29 |              25.0 – 29.9 |
| Obese         |            32.26 |              ≥ 30.0 |

The average BMI for each health status closely matches the corresponding WHO classification ranges, suggesting that the target variable is consistent with established medical standards.

| Variable       | Your Dataset         | Real-world Plausibility                 |
| -------------- | -------------------- | --------------------------------------- |
| Age            | 18–80 years          | Realistic                               |
| Height         | 145–198.6 cm         | Realistic                               |
| Weight         | 40–116.6 kg          | Realistic                               |
| BMI            | 12.9–44.0            | Realistic, though extremes are uncommon |
| Mean BMI       | 26.1                 | Very close to many adult populations    |
| Water Intake   | 1.5–5.0 L/day        | Plausible                               |
| Calorie Intake | 1,000–5,158 kcal/day | Broad but plausible                     |

Overall, the dataset contains values that fall within realistic human ranges and aligns well with WHO BMI standards. However, some characteristics suggest that it may be synthetically generated, including the absence of missing values and the presence of implausible observations such as negative carbohydrate intake. Consequently, the dataset is suitable for educational purposes and predictive modeling, but its ability to represent a real population should be interpreted with caution.

---

### Missing Values

In [15]:
df.isnull().sum()

Person_ID                    0
Age                          0
Gender                       0
Height_cm                    0
Weight_kg                    0
BMI                          0
Activity_Level               0
Daily_Calorie_Requirement    0
Daily_Calorie_Consumed       0
Protein_Intake_g             0
Carbohydrate_Intake_g        0
Fat_Intake_g                 0
Water_Intake_Liters          0
Diet_Type                    0
Health_Status                0
dtype: int64

No missing values were found in the dataset. All 6,000 records contain complete information across the 15 features.

### Duplicate Records Check

In [16]:
n_duplicated = df.duplicated().sum()

duplicate_pct = (
    n_duplicated / len(df) * 100
)

print(f"Exact duplicated rows before cleaning: {n_duplicated}")
print(f"Percentage of duplicated records: {duplicate_pct:.2f}%")

Exact duplicated rows before cleaning: 0
Percentage of duplicated records: 0.00%


No exact duplicate records were identified in the dataset, indicating that each observation represents a unique individual.

---

## Distribution Analysis

### Numeric

#### Histograms

In [17]:
# Select numeric columns
numeric_columns = df.select_dtypes(
    include=np.number
).columns

# Dashboard layout
n_cols = 4
n_rows = math.ceil(len(numeric_columns) / n_cols)

# Create subplot figure
fig = make_subplots(
    rows=n_rows,
    cols=n_cols,
    subplot_titles=numeric_columns
)

# Add histograms
for i, col in enumerate(numeric_columns):
    row = (i // n_cols) + 1
    col_pos = (i % n_cols) + 1

    fig.add_trace(
        go.Histogram(
            x=df[col],
            name=col,
            nbinsx=30
        ),
        row=row,
        col=col_pos
    )

# Update layout
fig.update_layout(
    title="Interactive Histogram Dashboard — Numeric Features",
    height=300 * n_rows,
    width=1400,
    showlegend=False,
    template="plotly_white",
    bargap=0.05
)

# Descriptive axis labels
fig.update_xaxes(title_text="Feature Values")
fig.update_yaxes(title_text="Number of Observations")

# Show dashboard
fig.show()

**Analysis**

- `Age` shows a relatively uniform distribution across the available age range.
- `Height_cm`, `Weight_kg`, `BMI`, `Daily_Calorie_Requirement`, and `Daily_Calorie_Consumed` exhibit approximately bell-shaped distributions, suggesting values are concentrated around the mean.
- `Protein_Intake_g` and `Fat_Intake_g` are positively skewed, with long right tails indicating some individuals have unusually high intake levels.
- `Carbohydrate_Intake_g` is roughly bell-shaped but slightly asymmetric, with a second peak toward lower values.
- `Water_Intake_Liters` is approximately bell-shaped with a slight skew toward higher consumption values.
- Several variables contain extreme observations, which may represent either valid outliers or data quality issues and should be examined further.

### Categorical

#### Categorical Value Consistency

In [18]:
categorical_cols = df.select_dtypes(
    include=["str", "category"]
).columns

for col in categorical_cols:
    print(f"\n{'='*40}")
    print(f"Column: {col}")
    print(f"{'='*40}")

    summary = (
        df[col]
        .value_counts(dropna=False)
        .reset_index()
    )

    summary.columns = [col, 'Count']

    summary['Percentage'] = (
        summary['Count'] / len(df) * 100
    ).round(2)

    print(summary)


Column: Person_ID
     Person_ID  Count  Percentage
0        P0001      1        0.02
1        P0002      1        0.02
2        P0003      1        0.02
3        P0004      1        0.02
4        P0005      1        0.02
...        ...    ...         ...
5995     P5996      1        0.02
5996     P5997      1        0.02
5997     P5998      1        0.02
5998     P5999      1        0.02
5999     P6000      1        0.02

[6000 rows x 3 columns]

Column: Gender
   Gender  Count  Percentage
0    Male   2892       48.20
1  Female   2855       47.58
2   Other    253        4.22

Column: Activity_Level
      Activity_Level  Count  Percentage
0     Lightly Active   1627       27.12
1  Moderately Active   1509       25.15
2          Sedentary   1487       24.78
3        Very Active    970       16.17
4            Athlete    407        6.78

Column: Diet_Type
       Diet_Type  Count  Percentage
0       Balanced   1536       25.60
1   High Protein   1076       17.93
2     Vegetarian   1055  

The category values are consistent across all categorical variables, with no apparent spelling errors or duplicate labels. The value **"Other"** appears as a valid category in the `Gender` variable.

In [19]:
MAX_CATEGORIES = 10

n_cols = 2
n_rows = math.ceil(len(categorical_cols) / n_cols)

fig = make_subplots(
    rows=n_rows,
    cols=n_cols,
    subplot_titles=categorical_cols,
    vertical_spacing=0.10
)

for i, col in enumerate(categorical_cols):

    summary = (
        df[col]
        .value_counts(dropna=False)
        .head(MAX_CATEGORIES)
        .reset_index()
    )

    summary.columns = [col, "Count"]

    row = (i // n_cols) + 1
    col_pos = (i % n_cols) + 1

    fig.add_trace(
        go.Bar(
            x=summary[col].astype(str),
            y=summary["Count"],
            text=summary["Count"],
            textposition="outside",
            name=col
        ),
        row=row,
        col=col_pos
    )

fig.update_layout(
    title="Categorical Variables Distribution Dashboard",
    height=400 * n_rows,
    width=1400,
    showlegend=False,
    template="plotly_white"
)

fig.show()

**Analysis**

- `Person_ID` is a unique identifier for each individual, confirming that it provides no predictive value and should be excluded from modeling.

- `Gender` is relatively balanced between **Male** (48.2%) and **Female** (47.6%), while **Other** represents only 4.2% of the records. This smaller group may be underrepresented in statistical analyses.

- `Activity_Level` is concentrated in the **Lightly Active** (27.1%), **Moderately Active** (25.2%), and **Sedentary** (24.8%) categories. **Athletes** represent only 6.8% of the dataset.

- `Diet_Type` is fairly diverse, with **Balanced** being the most common diet (25.6%). The remaining diet types are reasonably represented, although **Vegan** (9.8%) and **Keto** (12.0%) are less frequent.

- `Health_Status` is dominated by **Overweight** (41.7%) and **Healthy** (37.5%) individuals. **Obese** accounts for 17.9% of the records, while **Underweight** represents only 2.9%.

- The uneven distribution of **Health_Status** suggests a potential **class imbalance**, particularly for the **Underweight** category, which should be considered during model training and evaluation.

### Countplots

#### Outlier Detection

In [20]:
# Select numeric columns
numeric_columns = df.select_dtypes(
    include=np.number
).columns

if len(numeric_columns) > 0:

    # create boxplots
    # Dashboard layout
    n_cols = 4
    n_rows = math.ceil(len(numeric_columns) / n_cols)

    # Create subplot figure
    fig = make_subplots(
        rows=n_rows,
        cols=n_cols,
        subplot_titles=numeric_columns
    )

    # Add boxplots
    for i, col in enumerate(numeric_columns):
        row = (i // n_cols) + 1
        col_pos = (i % n_cols) + 1

        fig.add_trace(
            go.Box(
                y=df[col],
                name=col,
                boxmean=True
            ),
            row=row,
            col=col_pos
        )

    # Update layout
    fig.update_layout(
        title="Interactive Boxplot Dashboard — Numeric Features",
        height=300 * n_rows,
        width=1400,
        showlegend=False,
        template="plotly_white"
    )

    # Add axis labels
    fig.update_yaxes(title_text="Observed Values")

    # Show dashboard
    fig.show()

else:
    print("No numeric variables available for outlier analysis.")

**Analysis**

Most numerical variables contain outliers, while `Age` does not show significant extreme values. The detected outliers generally fall within plausible human ranges (e.g., height, weight, BMI, calorie intake, and water consumption) and may represent genuine individual differences rather than data errors.

Therefore, these observations should be retained and handled appropriately during preprocessing, as they may contain valuable information for predictive modeling. However, clearly invalid values, such as negative carbohydrate intake, should be investigated separately as potential data quality issues.

IQR calculation.

In [25]:
feature_columns = df.select_dtypes(
    include=np.number
).columns

outlier_summary = []

for col in feature_columns:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1

    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    n_outliers = ((df[col] < lower_bound) | (df[col] > upper_bound)).sum()

    outlier_summary.append({
        "Feature": col,
        "Outliers": n_outliers,
        "Percentage": round(n_outliers / len(df) * 100, 2)
    })

outlier_df = pd.DataFrame(outlier_summary)

outlier_df.sort_values(by='Percentage', ascending=False)

,Feature,Outliers,Percentage
8,Fat_Intake_g,499,8.32
7,Carbohydrate_Intake_g,246,4.10
6,Protein_Intake_g,188,3.13
9,Water_Intake_Liters,114,1.90
5,Daily_Calorie_Consumed,74,1.23
4,Daily_Calorie_Requirement,56,0.93
3,BMI,43,0.72
2,Weight_kg,12,0.20
1,Height_cm,4,0.07
0,Age,0,0.00


`Fat_Intake_g` has the highest proportion of outliers (**8.32%**), followed by `Carbohydrate_Intake_g` (**4.10%**) and `Protein_Intake_g` (**3.13%**). This suggests substantial variability in macronutrient consumption, likely reflecting the different diet types represented in the dataset.

The remaining variables show relatively few outliers, with less than **2%** of observations identified as extreme values. `Age` contains no outliers according to the IQR method.

Overall, the outlier percentages are low to moderate and appear consistent with natural variation in dietary and health-related measurements rather than widespread data quality issues. However, specific anomalous values, such as negative carbohydrate intake, should be investigated separately.

### Data Quality Issues

In [27]:
(df['Carbohydrate_Intake_g'] < 0).sum()

np.int64(54)

In [29]:
df.loc[
    df['Carbohydrate_Intake_g'] < 0,
    [
        'Person_ID',
        'Age',
        'Gender',
        'Diet_Type',
        'Carbohydrate_Intake_g',
        'Protein_Intake_g',
        'Fat_Intake_g',
        'Health_Status'
    ]
].sort_values(by='Carbohydrate_Intake_g', ascending=True)

,Person_ID,Age,Gender,Diet_Type,Carbohydrate_Intake_g,Protein_Intake_g,Fat_Intake_g,Health_Status
1885,P1886,33,Male,Keto,-23.6,187.8,214.0,Overweight
5570,P5571,29,Other,Keto,-22.7,186.6,217.1,Healthy
3112,P3113,39,Female,Keto,-19.2,197.3,217.0,Healthy
178,P0179,37,Female,Keto,-18.9,191.3,222.0,Overweight
3319,P3320,60,Female,Keto,-17.4,170.8,191.1,Overweight
2364,P2365,67,Female,Keto,-17.2,167.3,187.3,Overweight
2739,P2740,24,Male,Keto,-16.4,248.2,274.8,Overweight
4216,P4217,59,Male,Keto,-16.3,190.0,217.9,Healthy
5259,P5260,27,Female,Keto,-15.9,198.6,213.6,Healthy
4451,P4452,79,Female,Keto,-15.4,144.1,169.4,Overweight


A total of 54 records contained negative values for `Carbohydrate_Intake_g`. Since carbohydrate intake is measured as grams consumed and nutritional guidelines define very low-carbohydrate diets as intakes below 50 g/day rather than negative values, these observations are physiologically impossible and were considered data quality issues.

All affected records belonged to the Keto diet category, suggesting that the issue may have originated during data generation or preprocessing rather than representing real dietary behavior.

### Class Imbalance

The target variable is moderately imbalanced, with the Underweight class representing only 2.9% of observations. This should be considered during model training and evaluation.

---

# Exploratory Data Analysis (EDA)

## Correlation Analysis

In [22]:
numeric_columns = df.select_dtypes(
    include=np.number
).columns

corr_matrix = df[numeric_columns].corr()

corr_matrix

,Age,Height_cm,Weight_kg,BMI,Daily_Calorie_Requirement,Daily_Calorie_Consumed,Protein_Intake_g,Carbohydrate_Intake_g,Fat_Intake_g,Water_Intake_Liters
Age,1.000000,0.003546,0.105740,0.109039,-0.252781,-0.175597,-0.084040,-0.101428,-0.087563,-0.023625
Height_cm,0.003546,1.000000,0.413814,-0.246448,0.517498,0.364654,0.176652,0.218704,0.170418,0.077350
Weight_kg,0.105740,0.413814,1.000000,0.775497,0.506589,0.690917,0.326290,0.395661,0.351608,-0.056858
BMI,0.109039,-0.246448,0.775497,1.000000,0.178808,0.477787,0.223382,0.265503,0.254712,-0.115819
Daily_Calorie_Requirement,-0.252781,0.517498,0.506589,0.178808,1.000000,0.908845,0.430931,0.517020,0.465744,0.660469
Daily_Calorie_Consumed,-0.175597,0.364654,0.690917,0.477787,0.908845,1.000000,0.468970,0.570675,0.513449,0.522339
Protein_Intake_g,-0.084040,0.176652,0.326290,0.223382,0.430931,0.468970,1.000000,-0.132679,0.253324,0.246838
Carbohydrate_Intake_g,-0.101428,0.218704,0.395661,0.265503,0.517020,0.570675,-0.132679,1.000000,-0.300949,0.291004
Fat_Intake_g,-0.087563,0.170418,0.351608,0.254712,0.465744,0.513449,0.253324,-0.300949,1.000000,0.275897
Water_Intake_Liters,-0.023625,0.077350,-0.056858,-0.115819,0.660469,0.522339,0.246838,0.291004,0.275897,1.000000


### Heatmap visualization

In [23]:
# Correlation matrix
corr_matrix = df[numeric_columns].corr()

# Create mask for upper triangle
mask = np.triu(
    np.ones_like(corr_matrix, dtype=bool)
)

# Apply mask
corr_matrix_masked = corr_matrix.mask(mask)

fig = px.imshow(
    corr_matrix_masked,
    text_auto=".2f",
    color_continuous_scale="BuGn",
    aspect="auto",
    title="Correlation Matrix Heatmap"
)

fig.update_layout(
    width=900,
    height=800
)

fig.show()

**Analysis**

- A **very strong positive correlation** exists between **Daily_Calorie_Requirement** and **Daily_Calorie_Consumed** (**r = 0.91**), indicating that individuals with higher calorie needs tend to consume more calories.

- `BMI` shows a **strong positive correlation** with `Weight_kg` (**r = 0.78**), which is expected since weight is a key component of BMI calculation.

- `Daily_Calorie_Consumed` has a **strong correlation** with `Weight_kg` (**r = 0.69**) and a **moderate correlation** with `BMI` (**r = 0.48**), suggesting that higher calorie intake is associated with greater body weight and BMI.

- `Daily_Calorie_Requirement` is moderately correlated with both `Height_cm` (**r = 0.52**) and `Weight_kg` (**r = 0.51**), reflecting their role in estimating energy needs.

- `Carbohydrate_Intake_g` and `Fat_Intake_g` exhibit a moderate negative correlation (**r = -0.30**), likely reflecting differences among dietary patterns such as Keto and High-Carbohydrate diets.

- `Age` shows generally weak correlations with the other variables, suggesting that age alone is not a strong predictor of dietary intake or body composition in this dataset.

- No evidence of severe multicollinearity is observed beyond expected relationships among variables derived from similar physiological measures.

- It would be valuable to repeat the analysis after encoding the categorical variables and including `Health_Status`. This could reveal which dietary, demographic, and lifestyle factors are most strongly associated with the target variable.

## Feature Trends Across Health Status 

In [38]:
health_order = [
    'Underweight',
    'Healthy',
    'Overweight',
    'Obese'
]

features = [
    'BMI',
    'Weight_kg',
    'Daily_Calorie_Consumed',
    'Daily_Calorie_Requirement',
    'Protein_Intake_g',
    'Carbohydrate_Intake_g',
    'Fat_Intake_g',
    'Water_Intake_Liters'
]

mean_df = (
    df.groupby('Health_Status')[features]
        .mean()
        .reindex(health_order)
)

z_df = mean_df.apply(zscore)

long_df = z_df.reset_index().melt(
    id_vars='Health_Status',
    var_name='Feature',
    value_name='Standardized Mean'
)

fig = px.line(
    long_df,
    x='Health_Status',
    y='Standardized Mean',
    color='Feature',
    markers=True,
    title='Feature Trends Across Health Status'
)

fig.show()

**Analysis**

- Most numerical features show a clear trend across health status categories, generally increasing from **Underweight** to **Obese**.
- `BMI`, `Weight_kg`, `Daily_Calorie_Consumed`, and macronutrient intake exhibit the strongest positive trends, suggesting a relationship between higher energy intake and higher body weight.
- `Water_Intake_Liters` does not follow this pattern. Individuals classified as **Healthy** have the highest average water intake, while both underweight and obese groups tend to consume less water.
- These trends indicate that nutritional intake and body composition are closely associated with health status in the dataset, whereas water consumption appears to have a different relationship.

## Health Status Distribution by Diet Type

In [39]:
health_order = [
    'Underweight',
    'Healthy',
    'Overweight',
    'Obese'
]

cross = pd.crosstab(
    df['Diet_Type'],
    df['Health_Status'],
    normalize='index'
) * 100

cross = cross[health_order].reset_index()

long_df = cross.melt(
    id_vars='Diet_Type',
    var_name='Health_Status',
    value_name='Percentage'
)

fig = px.bar(
    long_df,
    x='Diet_Type',
    y='Percentage',
    color='Health_Status',
    category_orders={'Health_Status': health_order},
    title='Health Status Distribution by Diet Type',
    text_auto='.1f'
)

fig.update_layout(
    yaxis_title='Percentage (%)',
    barmode='stack'
)

fig.show()

**Analysis**

- The distribution of **Health_Status** is relatively similar across all diet types, with no diet showing a substantially different pattern from the others.
- In most diets, **Healthy** and **Overweight** individuals represent the largest groups, while **Underweight** remains the least frequent category.
- Although minor differences exist among diet types, there is no strong evidence that a particular diet is associated with a markedly better or worse health status in this dataset.
- These results suggest that **diet type alone may not be a strong predictor of health status**, and that other factors such as calorie intake, physical activity, and body composition may play a more important role.

## Health Status Distribution by Activity Level

In [ ]:
health_order = [
    'Underweight',
    'Healthy',
    'Overweight',
    'Obese'
]

activity_order = [
    'Sedentary',
    'Lightly Active',
    'Moderately Active',
    'Very Active',
    'Athlete'
]

# Percentage distribution
cross = (
    pd.crosstab(
        df['Activity_Level'],
        df['Health_Status'],
        normalize='index'
    ) * 100
)

cross = cross[health_order].reindex(activity_order)

plot_df = (
    cross.reset_index()
        .melt(
            id_vars='Activity_Level',
            var_name='Health_Status',
            value_name='Percentage'
        )
)

fig = px.bar(
    plot_df,
    x='Activity_Level',
    y='Percentage',
    color='Health_Status',
    barmode='group',  # grouped bars
    category_orders={
        'Activity_Level': activity_order,
        'Health_Status': health_order
    },
    text_auto='.1f',
    title='Health Status Distribution by Activity Level (%)'
)

fig.update_layout(
    yaxis_title='Percentage (%)',
    xaxis_title='Activity Level'
)

fig.show()

**Analysis**

- **Sedentary** and **Lightly Active** individuals show the highest proportions of **Overweight** cases.
- **Moderately Active**, **Very Active**, and **Athlete** groups have **Healthy** as their most common health status category.
- The proportion of **Obese** individuals decreases by approximately half from **Sedentary** to **Lightly Active** and continues to decline slightly as activity levels increase.
- **Very Active** individuals exhibit slightly better health status distributions than **Moderately Active** individuals.
- The **Athlete** group shows the most favorable distribution, with the highest percentage of **Healthy** individuals and the lowest percentages of **Overweight** and **Obese** cases.
- Overall, the results suggest that **higher levels of physical activity are associated with better health outcomes**, making **Activity_Level** a potentially important predictor of health status.

---

## Potential Target Variable

**Candidate Target Variable**

- `Health_Status`

**Potential ML Task**

- Classification

**Rationale**

`Health_Status` is a suitable target variable because it is influenced by demographic, biometric, lifestyle, and dietary factors such as BMI, physical activity level, calorie intake, macronutrient consumption, and water intake.

The exploratory analysis suggests that higher physical activity levels are associated with better health outcomes, while dietary and body composition variables also show meaningful relationships with health status.

A classification model could be developed to identify individuals at risk of being **Underweight**, **Overweight**, or **Obese**, supporting preventive healthcare and personalized nutrition recommendations.

In addition, the dataset could support a secondary regression task using `Daily_Calorie_Requirement` as the target variable to estimate individual calorie needs. Such models could help healthcare providers design personalized diet, exercise, and wellness programs aimed at improving health outcomes and encouraging healthier lifestyles.

---

# EDA Findings Summary

## Findings

- No missing values were found in the dataset.
- No duplicate records were identified.
- Most numerical variables show approximately bell-shaped distributions, while protein and fat intake are positively skewed.
- Outliers are present in most numerical features, particularly **Fat_Intake_g** (8.3%), **Carbohydrate_Intake_g** (4.1%), and **Protein_Intake_g** (3.1%). Most appear to be valid observations.
- A total of **54 records (0.9%)** contain negative carbohydrate intake values, which are physiologically impossible and should be treated as data quality issues.
- A very strong correlation exists between **Daily_Calorie_Requirement** and **Daily_Calorie_Consumed** (r = 0.91).
- **BMI** is strongly correlated with **Weight_kg** (r = 0.78), which is expected because weight is a component of the BMI formula.
- Health status shows clear relationships with physical activity level, with more active individuals generally exhibiting healthier outcomes.
- Diet type shows relatively small differences in health status distribution compared with activity level.
- The dataset values are generally realistic and align with WHO BMI classifications, although some characteristics suggest the dataset may be synthetically generated.

## Suitability for Machine Learning

- **Classification:** `Health_Status` is a suitable target variable because it is categorical and shows relationships with biometric, dietary, and lifestyle features.
- **Regression (optional):** `Daily_Calorie_Requirement` could be used as a continuous target variable to estimate individual calorie needs.

The dataset appears suitable for developing predictive models to support personalized nutrition recommendations and identify individuals at risk of unhealthy health outcomes.

---

# Appendix

## References

* Healthy Diet and Calorie Intake Dataset. Kaggle. Available at: [https://www.kaggle.com/datasets/aliyasaly1231/healthy-diet-and-calorie-intake](https://www.kaggle.com/datasets/aliyasaly1231/healthy-diet-and-calorie-intake)
* World Health Organization (WHO). Body Mass Index (BMI). Available at: [https://www.who.int/data/gho/data/themes/topics/topic-details/GHO/body-mass-index](https://www.who.int/data/gho/data/themes/topics/topic-details/GHO/body-mass-index)
* Valenzuela, R., et al. *Dieta baja en carbohidratos y dieta cetogénica: impacto en enfermedades metabólicas y reproductivas*. Revista Médica de Chile. Available at: [https://www.scielo.cl/article_plus.php?lng=es&pid=S0034-98872020001101630&tlng=es](https://www.scielo.cl/article_plus.php?lng=es&pid=S0034-98872020001101630&tlng=es)
* Machine Learning notebooks from the bootcamp repository (`sonda2026` branch).
* Machine Learning notebooks from the shared Assistanship Google Drive folder.
* Pandas Documentation. Available at: [https://pandas.pydata.org/docs/user_guide/index.html](https://pandas.pydata.org/docs/user_guide/index.html)
* Plotly Documentation. Available at: [https://plotly.com/python/](https://plotly.com/python/)

## Acknowledgements

I would like to thank my instructors for their guidance, continuous support, and encouragement throughout this project.

I also acknowledge the use of AI-assisted tools for support with debugging, code review, documentation, and concept exploration during the development of this analysis. All final decisions, interpretations, and conclusions presented in this work are my own.

### Dataset Limitations

Although the dataset contains realistic values and aligns with established BMI classifications, some characteristics (e.g., absence of missing values and the presence of implausible negative carbohydrate intake values) suggest that it may be partially synthetic. Therefore, results should be interpreted primarily as an educational and machine learning exercise rather than as evidence representative of a real population.